# Unity Gateway — governed access to LLMs

**Problem:** everyone needs LLMs, but ad-hoc API keys mean no permissions, no limits, no usage tracking.

**Unity Gateway:** every model becomes a **Unity Catalog object** (`system.ai.meta-llama-3-3-70b-instruct`, or your own `retail_ai_prod.models.<name>`), so AI access gets:

* **Permissions** — `EXECUTE` grants, like any table
* **Routing** — your own service can split traffic between models (A/B, shadow)
* **Limits & policies** — rate limits, budgets, guardrails per service
* **Observability** — usage in `system.ai_gateway.usage`

In [0]:
# Step 1: see which model services are available to us in system.ai
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

services = w.api_client.do(
    "GET",
    "/api/2.1/unity-catalog/model-services?parent=schemas/system.ai&view=BASIC",
)

print("Callable model services:")
for s in services.get("model_services", []):
    print(" -", s["name"].replace("model-services/", ""))

Callable model services:
 - system.ai.gpt-oss-120b
 - system.ai.meta-llama-3-1-8b-instruct
 - system.ai.bge-large-en
 - system.ai.llama-4-maverick
 - system.ai.qwen35-122b-a10b
 - system.ai.gemma-3-12b
 - system.ai.qwen3-embedding-0-6b
 - system.ai.gte-large-en
 - system.ai.meta-llama-3-3-70b-instruct
 - system.ai.gpt-oss-20b
 - system.ai.qwen3-next-80b-a3b-instruct


### Step 2 — Call a model through the gateway

One REST call — Chat Completions on the `/ai-gateway` route — with full Unity Catalog governance and our workspace credentials. No API keys.

In [0]:
# Step 2: call a model through the gateway — one REST call, full UC governance
import requests

MODEL = "system.ai.meta-llama-3-3-70b-instruct"

response = requests.post(
    f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1/chat/completions",
    headers={**w.config.authenticate(), "Content-Type": "application/json"},
    json={
        "model": MODEL,
        "messages": [
            {"role": "user", "content": (
                "Our top retail categories are Electronics, Fashion and Home & Kitchen. "
                "Suggest one loyalty promotion per category. One line each."
            )}
        ],
        "max_tokens": 200,
    },
    timeout=120,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])

Electronics: Offer 10% off on all purchases made within 30 days of a customer's previous electronics purchase.
Fashion: Launch a "Buy 2 Get 1 Free" loyalty program on select fashion brands for customers who have made at least 3 purchases in the last 6 months.
Home & Kitchen: Provide a $20 voucher to customers who spend $100 or more on Home & Kitchen products within a 60-day period, redeemable on their next purchase.


### Step 3 — LLMs inside SQL with `ai_query`

Batch inference straight from the sales table: one marketing pitch per product category, generated in a single SELECT.

In [0]:
%sql
-- Step 3: LLMs inside SQL — ai_query does batch inference on table data.
-- One marketing pitch per product category, straight from the sales table.
SELECT
  category,
  COUNT(*) AS transactions,
  ai_query(
    'system.ai.meta-llama-3-1-8b-instruct',
    concat('Write a one-sentence sales pitch for a retail category called "', category, '".')
  ) AS pitch
FROM retail_ai_prod.gold_data.sales_transactions
GROUP BY category
ORDER BY transactions DESC

category,transactions,pitch
Electronics,225,"""Upgrade your life with the latest and greatest in technology, from cutting-edge smartphones and laptops to innovative smart home devices and entertainment systems, all under one roof in our Electronics department."""
Home & Kitchen,184,"""Transform your living space into a haven of comfort and style with our vast selection of Home & Kitchen essentials, from modern furniture and appliances to cookware, decor, and more."""
Fashion,176,"""Step into the latest trends and express your personal style with our vast collection of on-trend clothing, shoes, and accessories that will elevate your wardrobe and make you feel confident and chic."""
Beauty,165,"""Unlock your full potential with our vast selection of beauty products and expert advice, helping you to enhance your natural beauty and feel confident in your own skin."""
Sports,163,"""Get ready to score big with our wide selection of athletic gear, apparel, and equipment from top brands, helping you perform at your best and look great doing it!"""
Grocery,87,"""Stock up on the essentials and indulge in your favorite treats with our wide selection of fresh produce, meats, dairy products, and household staples, all at affordable prices and convenient locations."""


### Step 4 — Create OUR OWN model service

`retail_ai_prod.models.retail_assistant`: a UC object that A/B-splits traffic — 70% Llama-3.3-70B, 30% GPT-OSS-120B shadow candidate. Idempotent: re-running reuses the existing service.

In [0]:
# Step 4: create OUR OWN model service with a traffic split:
# 70% of requests to Llama-3.3-70B (primary) + 30% to GPT-OSS-120B (shadow candidate)

SERVICE_NAME = "retail_ai_prod.models.retail_assistant"

def get_service(name):
    try:
        return w.api_client.do("GET", f"/api/2.1/unity-catalog/model-services/{name}")
    except Exception:
        return None

service = get_service(SERVICE_NAME)

if service is None:
    service = w.api_client.do(
        "POST",
        "/api/2.1/unity-catalog/model-services",
        query={"parent": "schemas/retail_ai_prod.models", "model_service_id": "retail_assistant"},
        body={
            "comment": "Retail assistant: 70% Llama-3.3-70B, 30% GPT-OSS-120B",
            "config": {
                "routing": {
                    "destinations": [
                        {
                            "name": "primary",
                            "destination_type": "DESTINATION_TYPE_PAY_PER_TOKEN_FOUNDATION_MODEL",
                            "pay_per_token_config": {
                                "model": "models/system.ai.llama_v3_3_70b_instruct"
                            },
                            "traffic_percentage": 70,
                        },
                        {
                            "name": "shadow",
                            "destination_type": "DESTINATION_TYPE_PAY_PER_TOKEN_FOUNDATION_MODEL",
                            "pay_per_token_config": {
                                "model": "models/system.ai.gpt-oss-120b"
                            },
                            "traffic_percentage": 30,
                        },
                    ]
                }
            },
        },
    )
    print("Created model service:", service["name"].replace("model-services/", ""))
else:
    print("Model service already exists — reusing:", service["name"].replace("model-services/", ""))

Created model service: retail_ai_prod.models.retail_assistant


### Step 5 — Call our service by its Unity Catalog name

Same API surface as the system models — the traffic split is invisible to callers. Request tags (`application`, `team`) make usage attributable in `system.ai_gateway.usage`.

In [0]:
# Step 5: call our service by its Unity Catalog name — the split is invisible to callers.
# Tag the request so usage tracking can attribute spend to this app.

response = requests.post(
    f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1/chat/completions",
    headers={
        **w.config.authenticate(),
        "Content-Type": "application/json",
        "Databricks-Ai-Gateway-Request-Tags": '{"application": "retail_demo", "team": "marketing"}',
    },
    json={
        "model": SERVICE_NAME,
        "messages": [
            {"role": "user", "content": "Give me 3 bullet points on why a retail chain needs churn prediction."}
        ],
        "max_tokens": 250,
    },
    timeout=120,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])

Here are three bullet points on why a retail chain needs churn prediction:

* **Revenue Protection**: Churn prediction helps retail chains identify customers who are likely to stop shopping with them, allowing them to take proactive measures to retain these high-value customers and prevent revenue loss. By targeting retention efforts at the right customers, retailers can minimize the financial impact of churn.
* **Personalized Marketing**: Churn prediction enables retail chains to create targeted marketing campaigns that address the specific needs and concerns of at-risk customers. By analyzing customer behavior and preferences, retailers can develop personalized offers and promotions that encourage loyalty and prevent churn.
* **Data-Driven Decision Making**: Churn prediction provides retail chains with valuable insights into customer behavior and preferences, allowing them to make data-driven decisions about marketing, sales, and customer service strategies. By analyzing churn patter

## What just happened

* Called foundation models through **Unity Gateway** — Python, SQL (`ai_query`), same governed path
* Created our **own UC model service** that A/B-splits traffic between two models (70/30)
* Tagged requests for **usage attribution** in `system.ai_gateway.usage`

**Governance in one line:** instead of sharing an API key, you grant `EXECUTE` on a UC object.